## ClinIQ AI — Expansion
Upgrades over `development.ipynb`:
- Dense semantic retrieval (sentence-transformers + FAISS) replacing TF-IDF cosine similarity
- Improved intent classifier with broader keyword coverage across 5 classes, retrained on raw question text so LIME attributions are meaningful
- Real XAI via LIME on the retrained intent classifier
- Answer text preserved with punctuation for readable display
- Side-by-side comparison of v1 (TF-IDF) vs v2 (dense) retrieval

Run all cells top to bottom. Saves new model files to `data/` with `_v2` suffix so original models are untouched.

### Cell 1 — Install new dependencies

In [1]:
import subprocess
subprocess.run([
    'pip', 'install',
    'sentence-transformers', 'faiss-cpu', 'lime'
], capture_output=True)
print('done')

done


### Cell 2 — Imports

In [2]:
import pandas as pd
import numpy as np
import joblib
import re
import faiss
from sentence_transformers import SentenceTransformer
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity as sklearn_cosine
from lime.lime_text import LimeTextExplainer
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
import warnings
warnings.filterwarnings('ignore')
print('imports done')

imports done


### Cell 3 — Load dataset and assign improved intent labels

In [3]:
df = pd.read_pickle('data/cliniq_data.pkl')
print(f'dataset: {len(df)} rows')
print(f'columns: {df.columns.tolist()}')
print(f'\noriginal intent distribution:')
print(df['intent'].value_counts())

# original assign_intent had 3 narrow keyword lists
# 'what are the symptoms of diabetes' was getting general_info
# because it only matched if 'symptom' was literally present
# expanded to 5 classes with broader coverage

def assign_intent_v2(question):
    q = question.lower()
    symptom_kws = [
        'symptom', 'sign', 'feel', 'pain', 'ache', 'fever', 'cough',
        'fatigue', 'tired', 'nausea', 'vomit', 'dizziness', 'dizzy',
        'swelling', 'rash', 'bleeding', 'bruising', 'shortness of breath',
        'chest pain', 'headache', 'weak', 'loss of', 'difficulty',
        'problem', 'trouble', 'discomfort', 'what does it feel',
        'how does it feel', 'look like', 'notice'
    ]
    treatment_kws = [
        'treat', 'cure', 'medication', 'drug', 'therapy', 'medicine',
        'dose', 'dosage', 'surgery', 'procedure', 'operation', 'manage',
        'control', 'remedy', 'heal', 'recover', 'antibiotic', 'vaccine',
        'injection', 'prescription', 'pill', 'tablet', 'supplement',
        'how to treat', 'how to cure', 'how is it treated', 'what helps'
    ]
    cause_kws = [
        'cause', 'why', 'reason', 'trigger', 'lead to', 'result in',
        'develop', 'origin', 'how do you get', 'how does one get',
        'what causes', 'risk factor'
    ]
    prevention_kws = [
        'prevent', 'avoid', 'reduce risk', 'protect', 'lower chance',
        'stop from', 'keep from', 'lower risk', 'how to avoid'
    ]
    if any(w in q for w in symptom_kws):
        return 'symptom_query'
    elif any(w in q for w in treatment_kws):
        return 'treatment_query'
    elif any(w in q for w in cause_kws):
        return 'cause_query'
    elif any(w in q for w in prevention_kws):
        return 'prevention_query'
    else:
        return 'general_info'

df['intent_v2'] = df['question'].apply(assign_intent_v2)
print(f'\nimproved intent distribution:')
print(df['intent_v2'].value_counts())

dataset: 14301 rows
columns: ['question', 'answer', 'source', 'focus_area', 'intent', 'question_clean', 'question_processed', 'answer_clean']

original intent distribution:
intent
general_info       9439
symptom_query      2817
treatment_query    2045
Name: count, dtype: int64

improved intent distribution:
intent_v2
general_info        8452
symptom_query       2898
treatment_query     2140
cause_query          651
prevention_query     160
Name: count, dtype: int64


### Cell 4 — Retrain intent classifier on raw question text

In [4]:
# KEY FIX: train on raw question text, not question_processed
# previous version trained on lemmatized text but LIME perturbs raw text
# that mismatch caused wrong predictions and near-zero LIME weights
# training on raw text means what LIME tests is what the model was trained on

X = df['question'].astype(str)   # raw questions, not question_processed
y = df['intent_v2']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

intent_vectorizer_v2 = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=15000,
    sublinear_tf=True
)

intent_model_v2 = LogisticRegression(
    max_iter=1000,
    C=1.0,
    class_weight='balanced'
)

intent_pipeline_v2 = make_pipeline(intent_vectorizer_v2, intent_model_v2)
intent_pipeline_v2.fit(X_train, y_train)

y_pred = intent_pipeline_v2.predict(X_test)
print(classification_report(y_test, y_pred))

                  precision    recall  f1-score   support

     cause_query       1.00      0.96      0.98       130
    general_info       0.99      1.00      0.99      1691
prevention_query       0.82      1.00      0.90        32
   symptom_query       1.00      0.97      0.98       580
 treatment_query       0.97      0.99      0.98       428

        accuracy                           0.99      2861
       macro avg       0.96      0.98      0.97      2861
    weighted avg       0.99      0.99      0.99      2861



### Cell 5 — Fix answer display (preserve punctuation)

In [5]:
# original answer_clean was produced by clean_text() which strips all punctuation
# answers displayed as unpunctuated walls of text
# fix: light cleaning only — remove URLs/HTML, keep sentence structure intact

def clean_answer(text):
    text = str(text)
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['answer_display'] = df['answer'].apply(clean_answer)

print('BEFORE (answer_clean — no punctuation):')
print(df['answer_clean'].iloc[0][:200])
print('\nAFTER (answer_display — punctuation preserved):')
print(df['answer_display'].iloc[0][:200])

BEFORE (answer_clean — no punctuation):
glaucoma is a group of diseases that can damage the eyes optic nerve and result in vision loss and blindness while glaucoma can strike anyone the risk is much greater for people over 60 how glaucoma d

AFTER (answer_display — punctuation preserved):
Glaucoma is a group of diseases that can damage the eye's optic nerve and result in vision loss and blindness. While glaucoma can strike anyone, the risk is much greater for people over 60. How Glauco


### Cell 6 — Build dense embeddings

In [6]:
# all-MiniLM-L6-v2: fast, 80MB, strong on medical text
# encodes meaning not just words — fixes vocabulary mismatch
# 'hypertension' will match 'high blood pressure', 'cardiac arrest' matches 'heart attack'

embed_model = SentenceTransformer('all-MiniLM-L6-v2')

print('encoding questions... (~2-3 min for 14k rows)')
questions = df['question'].astype(str).tolist()
question_embeddings = embed_model.encode(
    questions,
    batch_size=64,
    show_progress_bar=True
)
print(f'embeddings shape: {question_embeddings.shape}')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


encoding questions... (~2-3 min for 14k rows)


Batches:   0%|          | 0/224 [00:00<?, ?it/s]

embeddings shape: (14301, 384)


### Cell 7 — Build FAISS index

In [7]:
dim = question_embeddings.shape[1]  # 384
norms = np.linalg.norm(question_embeddings, axis=1, keepdims=True)
embeddings_norm = (question_embeddings / norms).astype('float32')

index = faiss.IndexFlatIP(dim)  # inner product == cosine sim after normalisation
index.add(embeddings_norm)

print(f'FAISS index built: {index.ntotal} vectors')

FAISS index built: 14301 vectors


### Cell 8 — Save all v2 artifacts

In [18]:
faiss.write_index(index, 'data/faiss_index_v2.bin')
np.save('data/question_embeddings_v2.npy', question_embeddings)
df.to_pickle('data/cliniq_data_v2.pkl')
joblib.dump(intent_pipeline_v2, 'data/intent_pipeline_v2.pkl')

print('saved: data/faiss_index_v2.bin')
print('saved: data/question_embeddings_v2.npy')
print('saved: data/cliniq_data_v2.pkl')
print('saved: data/intent_pipeline_v2.pkl')

saved: data/faiss_index_v2.bin
saved: data/question_embeddings_v2.npy
saved: data/cliniq_data_v2.pkl
saved: data/intent_pipeline_v2.pkl


### Cell 9 — Dense retrieval function

In [19]:
def dense_retrieve(user_query, top_k=3, threshold=0.45):
    query_emb = embed_model.encode([user_query])
    query_norm = (query_emb / np.linalg.norm(query_emb)).astype('float32')
    scores, indices = index.search(query_norm, top_k)
    scores, indices = scores[0], indices[0]
    best_score = float(scores[0])

    if best_score < threshold:
        return {
            'status': 'no_match',
            'message': "I couldn't find a confident answer in my medical database. Try rephrasing, or consult a healthcare professional.",
            'best_attempt': {
                'question': df.iloc[indices[0]]['question'],
                'similarity': best_score
            }
        }

    results = []
    for score, idx in zip(scores, indices):
        results.append({
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer_display'],
            'focus_area': df.iloc[idx]['focus_area'],
            'intent': df.iloc[idx]['intent_v2'],
            'similarity': float(score),
            'index': int(idx)
        })
    return {'status': 'match_found', 'results': results}

### Cell 10 — Test dense retrieval

In [20]:
test_queries = [
    'what are the symptoms of diabetes',
    'hypertension risks',
    'heart attack signs',
    'what is the best pizza topping',
    'kidney failure treatment',
]

for q in test_queries:
    out = dense_retrieve(q)
    print(f'\nQUERY: {q}')
    print(f'STATUS: {out["status"]}')
    if out['status'] == 'match_found':
        top = out['results'][0]
        print(f'MATCHED: {top["question"]}')
        print(f'SIM: {top["similarity"]:.4f}')
        print(f'ANSWER PREVIEW: {top["answer"][:150]}')
    else:
        print(f'CLOSEST: {out["best_attempt"]["question"]} ({out["best_attempt"]["similarity"]:.4f})')


QUERY: what are the symptoms of diabetes
STATUS: match_found
MATCHED: What are the symptoms of Diabetes ?
SIM: 0.9386
ANSWER PREVIEW: Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild tha

QUERY: hypertension risks
STATUS: match_found
MATCHED: Who is at risk for High Blood Pressure? ?
SIM: 0.7255
ANSWER PREVIEW: Not a Normal Part of Aging Nearly 1 in 3 American adults have high blood pressure. Many people get high blood pressure as they get older. However, get

QUERY: heart attack signs
STATUS: match_found
MATCHED: What are the symptoms of Heart Attack ?
SIM: 0.8382
ANSWER PREVIEW: Symptoms Can Vary Not all heart attacks begin with the sudden, crushing chest pain that often is shown on TV or in the movies. The warning signs and s

QUERY: what is the best pizza topping
STATUS: no_match
CLOSEST: What to do for Cystocele ? (0.2591)

QUERY: kidney failure treatment
STATUS: match_found
MATCHED:

### Cell 11 — LIME explainability on retrained intent classifier

In [21]:
lime_explainer = LimeTextExplainer(
    class_names=list(intent_pipeline_v2.classes_)
)

def explain_intent(user_query, num_features=6):
    exp = lime_explainer.explain_instance(
        user_query,
        intent_pipeline_v2.predict_proba,
        num_features=num_features,
        labels=list(range(len(intent_pipeline_v2.classes_)))
    )
    predicted_class = intent_pipeline_v2.predict([user_query])[0]
    class_idx = list(intent_pipeline_v2.classes_).index(predicted_class)
    attributions = exp.as_list(label=class_idx)
    attributions.sort(key=lambda x: abs(x[1]), reverse=True)
    return attributions

# test across all 5 intent classes
test_lime_queries = [
    'what are the symptoms of diabetes',      # symptom_query
    'how do you treat high blood pressure',   # treatment_query
    'what causes kidney disease',             # cause_query
    'how to prevent heart disease',           # prevention_query
    'what is glaucoma',                       # general_info
]

for q in test_lime_queries:
    predicted = intent_pipeline_v2.predict([q])[0]
    attributions = explain_intent(q)
    print(f'\nQUERY: {q}')
    print(f'PREDICTED INTENT: {predicted}')
    print('LIME attributions:')
    for word, weight in attributions[:5]:
        bar = '+' if weight > 0 else '-'
        print(f'  [{bar}] {word}: {weight:.4f}')


QUERY: what are the symptoms of diabetes
PREDICTED INTENT: symptom_query
LIME attributions:
  [+] symptoms: 0.2447
  [+] of: 0.1106
  [+] the: 0.0697
  [-] diabetes: -0.0530
  [+] are: 0.0513

QUERY: how do you treat high blood pressure
PREDICTED INTENT: general_info
LIME attributions:
  [-] pressure: -0.0920
  [+] do: 0.0856
  [-] blood: -0.0629
  [+] how: 0.0417
  [-] you: -0.0117

QUERY: what causes kidney disease
PREDICTED INTENT: cause_query
LIME attributions:
  [+] causes: 0.8120
  [-] kidney: -0.0410
  [+] what: 0.0388
  [-] disease: -0.0298

QUERY: how to prevent heart disease
PREDICTED INTENT: prevention_query
LIME attributions:
  [+] prevent: 0.5057
  [+] to: 0.1749
  [+] how: 0.1677
  [-] disease: -0.0436
  [+] heart: 0.0048

QUERY: what is glaucoma
PREDICTED INTENT: general_info
LIME attributions:
  [+] is: 0.5065
  [+] glaucoma: 0.0799
  [+] what: 0.0496


### Cell 12 — Full pipeline v2

In [22]:
def confidence_label(score, threshold=0.45):
    if score < threshold:
        return 'No reliable match', '🔴'
    elif score < 0.6:
        return 'Medium', '🟡'
    else:
        return 'High', '🟢'


def cliniq_pipeline_v2(user_query, threshold=0.45):
    if not user_query or not user_query.strip():
        return {'status': 'invalid_input', 'message': 'Please enter a question.'}

    predicted_intent = intent_pipeline_v2.predict([user_query])[0]
    intent_explanation = explain_intent(user_query)
    retrieval_result = dense_retrieve(user_query, threshold=threshold)

    if retrieval_result['status'] == 'no_match':
        score = retrieval_result['best_attempt']['similarity']
        label, emoji = confidence_label(score, threshold)
        return {
            'status': 'no_match',
            'predicted_intent': predicted_intent,
            'intent_explanation': intent_explanation,
            'confidence_label': label,
            'confidence_emoji': emoji,
            'similarity': score,
            'message': retrieval_result['message'],
            'closest_question': retrieval_result['best_attempt']['question']
        }

    results = retrieval_result['results']
    best = results[0]
    label, emoji = confidence_label(best['similarity'], threshold)

    return {
        'status': 'match_found',
        'predicted_intent': predicted_intent,
        'intent_explanation': intent_explanation,
        'confidence_label': label,
        'confidence_emoji': emoji,
        'similarity': round(best['similarity'], 4),
        'matched_question': best['question'],
        'focus_area': best['focus_area'],
        'answer': best['answer'],
        'alternative_matches': [
            {'question': r['question'], 'similarity': round(r['similarity'], 4)}
            for r in results[1:]
        ]
    }


out = cliniq_pipeline_v2('what are the symptoms of diabetes')
print('STATUS:', out['status'])
print('INTENT:', out['predicted_intent'])
print('LIME:', out['intent_explanation'])
if out['status'] == 'match_found':
    print('MATCHED Q:', out['matched_question'])
    print('SIM:', out['similarity'])
    print('ANSWER PREVIEW:', out['answer'][:200])

STATUS: match_found
INTENT: symptom_query
LIME: [(np.str_('symptoms'), 0.24245625388567582), (np.str_('of'), 0.11037578659170856), (np.str_('the'), 0.07155591282627304), (np.str_('are'), 0.05742704818183505), (np.str_('diabetes'), -0.05556766750079347), (np.str_('what'), 0.03349216819642779)]
MATCHED Q: What are the symptoms of Diabetes ?
SIM: 0.9386
ANSWER PREVIEW: Diabetes is often called a "silent" disease because it can cause serious complications even before you have symptoms. Symptoms can also be so mild that you dont notice them. An estimated 8 million peo


### Cell 13 — v1 vs v2 retrieval comparison

In [23]:
# loads original TF-IDF retrieval models saved by development.ipynb
# no need to re-run development.ipynb — just reads saved pkl files

retrieval_vectorizer_v1 = joblib.load('data/retrieval_vectorizer.pkl')
question_vectors_v1 = joblib.load('data/question_vectors.pkl')
df_v1 = pd.read_pickle('data/cliniq_data.pkl')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))
stop_words -= {'no', 'not', 'nor', 'never', 'without', 'more', 'less', 'above', 'below'}

def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+', '', text)
    text = re.sub(r'<[^>]+>', '', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[^a-z0-9\s\-]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t) > 2]
    return ' '.join(tokens)

def retrieve_v1(user_query):
    processed = preprocess_text(user_query)
    vec = retrieval_vectorizer_v1.transform([processed])
    sims = sklearn_cosine(vec, question_vectors_v1).flatten()
    idx = sims.argmax()
    return df_v1.iloc[idx]['question'], float(sims[idx])

test_queries = [
    'what are the symptoms of diabetes',
    'hypertension risks',
    'cardiac arrest warning signs',
    'renal failure causes',
    'how do I manage high blood sugar',
    'tell me about glaucoma',
    'what is the best pizza topping',
    'pain in chest when breathing',
    'what drugs are used for asthma',
    'autosomal recessive inheritance pattern',
]

print(f'{"QUERY":<45} {"V1 MATCH":<45} {"V1 SIM":>7}   {"V2 MATCH":<45} {"V2 SIM":>7}')
print('-' * 155)

for q in test_queries:
    v1_match, v1_sim = retrieve_v1(q)
    v1_short = v1_match[:42] + '...' if len(v1_match) > 45 else v1_match

    v2_result = dense_retrieve(q, top_k=1)
    if v2_result['status'] == 'match_found':
        v2_match = v2_result['results'][0]['question']
        v2_sim = v2_result['results'][0]['similarity']
    else:
        v2_match = 'NO MATCH'
        v2_sim = v2_result['best_attempt']['similarity']
    v2_short = v2_match[:42] + '...' if len(v2_match) > 45 else v2_match

    q_short = q[:42] + '...' if len(q) > 45 else q
    print(f'{q_short:<45} {v1_short:<45} {v1_sim:>7.4f}   {v2_short:<45} {v2_sim:>7.4f}')

QUERY                                         V1 MATCH                                       V1 SIM   V2 MATCH                                       V2 SIM
-----------------------------------------------------------------------------------------------------------------------------------------------------------
what are the symptoms of diabetes             What are the symptoms of Diabetes ?            1.0000   What are the symptoms of Diabetes ?            0.9386
hypertension risks                            What is (are) Portal hypertension ?            0.8442   Who is at risk for High Blood Pressure? ?      0.7255
cardiac arrest warning signs                  What is (are) Cardiac Arrest ?                 0.8325   How to diagnose Sudden Cardiac Arrest ?        0.8013
renal failure causes                          What causes Heart Failure ?                    0.4041   What causes What I need to know about Kidn...  0.8159
how do I manage high blood sugar              What is (are) High